In [104]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px

In [3]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

# MAIN

In [98]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("CUSTOM_hg38_episign/meth_matrix.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
epiSize = {}
for epiSign in [x for x in X.columns if x not in ('coord')]:
    epiSize[epiSign] = sum(X[epiSign] > 99)  # Catch 100% methyl

## Pre-processing

In [109]:
# Transpose
X_t = X.T  # Required
print(X_t.index)

to_PCA = X_t
if 'epiSize' in X_t.columns:
    to_PCA = X_t.drop('epiSize', axis=1)

# Remove 2nd row = size of epiSignormalize) then normalize
X_scaled = StandardScaler().fit_transform(to_PCA)

Index(['ADCADN.bed', 'ATRX.bed', 'AUTS18.bed', 'BAFopathy.bed', 'BFLS.bed',
       'CHARGE.bed', 'CdLS.bed', 'Down.bed', 'Dup7.bed', 'EEOC.bed',
       'FLHS.bed', 'GTPTS.bed', 'HMA.bed', 'HVDAS_C.bed', 'HVDAS_T.bed',
       'ICF1.bed', 'ICF2_3_4.bed', 'KDVS.bed', 'Kabuki.bed', 'Kleefstra.bed',
       'MRD51.bed', 'MRX93.bed', 'MRX97.bed', 'MRXCJS.bed', 'MRXSN.bed',
       'MRXSSR.bed', 'RMNS.bed', 'RSTS.bed', 'SBBYSS.bed', 'SETD1B.bed',
       'Sotos.bed', 'TBRS.bed', 'WDSTS.bed', 'Williams.bed', 'HG002_combined',
       'barcode04_combined'],
      dtype='object')


## PCA

In [110]:
# Run PCA:
NB_COMPON = 3
pca = PCA(n_components=NB_COMPON)
pcs = pca.fit_transform(X_scaled)

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

In [111]:
# Top N features of each componennt
compon_0_top = np.abs(pca.components_[0]).argsort()[::-1][:5]
print("Component 0:", list(X.index[compon_0_top]))

compon_1_top = np.abs(pca.components_[1]).argsort()[::-1][:5]
print("Component 1:", list(X.index[compon_1_top]))

Component 0: ['11:1272175-1272176', '12:50282921-50282922', '1:228471391-228471392', '5:13810085-13810086', '3:11705290-11705291']
Component 1: ['10:122149940-122149941', '11:126465987-126465988', '16:983407-983408', '5:73499355-73499356', '17:7241047-7241048']


In [112]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
print(pcs_df.loc[['Kabuki.bed', 'barcode04_combined', 'HG002_combined']])

                      compon0    compon1    compon2
Kabuki.bed          -3.977544  59.779906  26.873432
barcode04_combined   6.904513  -2.389131  -1.394341
HG002_combined      87.778581   1.455497  -0.505519


In [113]:
# Plot PCA
x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=epiSize,
        labels={x_compon:':'.join([x_compon,dict_compon[x_compon]]), y_compon:':'.join([y_compon,dict_compon[y_compon]])}
)
fig.show()

## t-SNE

In [114]:
# Run t-SNE:
# MEMOs:
# - Use 'perplex=2' as in Joris' paper
# - t-SNE is stochastic -> re-run multiple times ?
#
tsne = TSNE(n_components=2, perplexity=2).fit_transform(X_scaled)

In [128]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
tsne_df = pd.DataFrame(
    tsne,
    index=X.columns,
    columns=('compon0', 'compon1')
)
print(tsne_df.loc[['Kabuki.bed', 'barcode04_combined', 'HG002_combined']])

                       compon0     compon1
Kabuki.bed          126.051514  300.989075
barcode04_combined  -19.856714  -22.520119
HG002_combined      -69.960281  -53.077427


In [125]:
# Plot t-SNE
fig = px.scatter(
        x=tsne[:, 0],
        y=tsne[:, 1],
        hover_data=[pcs_df.index],
        color=epiSize
)
fig.show()